# Бенчмарк линейных baseline и MOMENT vs TimeXer

Параллельно с основным `training_pipeline.ipynb` (TimeXer) этот ноутбук
прогоняет три альтернативные архитектуры на тех же данных и считает
сравнимые метрики:

1. **VLinear** — vanilla Linear baseline из Zeng et al. (AAAI 2023,
   arXiv:2205.13504). Простейший возможный baseline класса LTSF-Linear:
   одна линейка `Linear(seq_len -> 1)` поканально + голова. Никакой
   декомпозиции, никаких трюков.
2. **XLinear** (arXiv:2601.09237, AAAI 2026) — MLP-baseline с
   sigmoid-gated variate-mixer и поддержкой экзогенных каналов.
3. **MOMENT** (Goswami et al., ICML 2024) — pretrained foundation model
   из HuggingFace `AutonLab/MOMENT-1-base`. Encoder заморожен,
   обучается только небольшая голова. При нашем размере датасета это
   ключевая защита от prediction collapse.

Идея: построить датасет один раз, переиспользовать для всех трёх
моделей, в конце — единая таблица-сравнение.


## 0. Окружение

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    if PROJECT_ROOT.exists():
        print(f'{PROJECT_ROOT} уже существует - пропускаем клонирование.')
    else:
        env = os.environ.copy()
        env['GIT_TERMINAL_PROMPT'] = '0'
        subprocess.run(
            [
                'git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
                f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git',
                str(PROJECT_ROOT),
            ],
            check=True,
            env=env,
        )
        print(f'Репозиторий склонирован в {PROJECT_ROOT}')
else:
    # Локальный Jupyter: ноутбук в notebooks/, корень - папка выше.
    PROJECT_ROOT = Path.cwd().parent

PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'IN_COLAB:     {IN_COLAB}')


In [ ]:
if IN_COLAB:
    print('Устанавливаем базовые зависимости...')
    subprocess.run(
        [
            'pip', 'install', '--quiet',
            'torch>=2.2', 'pandas>=2.1', 'pyarrow>=15.0',
            'scikit-learn>=1.4', 'requests>=2.31', 'yfinance>=0.2.40',
            'pydantic>=2.6', 'tqdm>=4.66', 'ipywidgets>=8.0',
        ],
        check=True,
    )
    print('Готово.')
    print()
    print('Устанавливаем momentfm для architecture=moment...')
    proc = subprocess.run(
        ['pip', 'install', 'momentfm', 'transformers', 'huggingface_hub'],
        capture_output=True, text=True, check=False,
    )
    if proc.returncode != 0:
        print('!! momentfm НЕ установился. Вывод pip:')
        print(proc.stdout[-2000:])
        print(proc.stderr[-2000:])
    else:
        last = (proc.stdout.strip().split(chr(10)))[-1]
        print(last if last else 'OK')
else:
    print('Локальный режим: используются зависимости из текущего окружения.')


In [ ]:
import sys

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import dataclasses
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from graduate_work.config import default_config, ModelConfig

cfg = default_config()
print('Тикеры:', cfg.data.tickers)
print('Горизонты:', cfg.data.horizons)
print('Окно:', cfg.data.window_size)
print('Mode:', cfg.data.mode)


## 1. Drive + данные

Подтягиваем уже скачанные через `training_pipeline.ipynb` CSV (моекс,
индексы, макро). Если их нет на Drive — нужно сначала запустить тот
блокнот.

In [ ]:
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

    drive_raw = DRIVE_BASE / 'data' / 'raw'
    if drive_raw.exists():
        shutil.copytree(drive_raw, cfg.paths.data_raw, dirs_exist_ok=True)
        n_files = sum(1 for _ in cfg.paths.data_raw.rglob('*.csv'))
        print(f'Подтянули с Drive {n_files} CSV в {cfg.paths.data_raw}')
    else:
        print(f'WARNING: на Drive нет {drive_raw} — сначала запусти training_pipeline.ipynb')


## 2. Сборка датасета (один раз)

In [ ]:
from graduate_work.features import build_dataset

prepared = build_dataset(
    cfg.data, cfg.paths,
    persist=True,
    trading_cfg=cfg.trading,
)
print('Число фич:', len(prepared.feature_cols))
print('Целевые колонки:', prepared.target_cols)
print(f"train: {prepared.train['x'].shape}  val: {prepared.val['x'].shape}  "
      f"test: {prepared.test['x'].shape}")

# P(UP) per horizon — уже в P(UP) видно дисбаланс, который требует pos_weight.
print()
print('=== P(UP) per horizon ===')
for split_name, split in (('train', prepared.train), ('val', prepared.val), ('test', prepared.test)):
    y = split['y']
    line = f'  {split_name:>5s}:'
    for j, hz in enumerate(cfg.data.horizons):
        line += f'  h={hz:>2d}={float((y[:, j] >= 0.5).mean()):.3f}'
    print(line)


## 3. Универсальный train + evaluate цикл

Helper-функция строит модель по `architecture`, обучает Trainer'ом
(автоматически вычисляющим `pos_weight`), запускает MC-инференс,
прогоняет Bayes/Conformal сигналы, делает бэктест и random-monkeys
3σ-тест.

In [ ]:
from graduate_work.backtest import compute_metrics, run_backtest, run_random_portfolios
from graduate_work.backtest.engine import prices_from_full_frame
from graduate_work.model import build_model, set_mc_dropout
from graduate_work.strategy import (
    ConformalSignalGenerator,
    SignalGenerator,
    attach_actual_targets,
    attach_lr_targets,
    build_predictions_frame,
    calibrate_bayes_threshold,
)
from graduate_work.training import Trainer, mc_predict, set_seed


# Готовим test_prices один раз — один и тот же бэктест-бэкенд для всех.
test = prepared.test
val = prepared.val
full = prepared.full_frame.copy()
if not isinstance(full.index, pd.DatetimeIndex):
    full.index = pd.to_datetime(full.index, utc=True)
test_start = pd.to_datetime(min(test['timestamp']), utc=True)
buffer = cfg.data.bar_timedelta * max(cfg.data.horizons)
test_end = pd.to_datetime(max(test['timestamp']), utc=True) + buffer
test_prices = prices_from_full_frame(
    full.loc[(full.index >= test_start) & (full.index <= test_end)],
)
bars_per_year = cfg.data.bars_per_year


def train_and_evaluate(arch: str, *, model_overrides: dict | None = None,
                       train_overrides: dict | None = None) -> dict:
    """Обучить и оценить одну архитектуру. Возвращает dict с метриками."""
    print(f'\n{"="*70}')
    print(f'>>> {arch.upper()}')
    print(f'{"="*70}')

    set_seed(cfg.training.seed)
    model_cfg = dataclasses.replace(
        cfg.model,
        architecture=arch,
        **(model_overrides or {}),
    )
    train_cfg = dataclasses.replace(cfg.training, **(train_overrides or {}))

    # 1) Build + fit.
    model = build_model(
        input_dim=prepared.num_features,
        num_horizons=len(cfg.data.horizons),
        cfg=model_cfg,
    )
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Параметров всего: {n_params:,};  обучаемых: {n_trainable:,}')

    trainer = Trainer(model, train_cfg, data_cfg=cfg.data, trading_cfg=cfg.trading)
    history = trainer.fit(prepared.train, prepared.val)

    n_ep = len(history.train_loss)
    best_ep = int(np.argmin(history.val_loss)) + 1 if n_ep else 0
    best_val = float(min(history.val_loss)) if n_ep else float('nan')
    print(f'Best epoch: {best_ep}/{n_ep}, val_loss={best_val:.5f}')

    # 2) MC-inference на test и val.
    is_classification = cfg.data.mode == 'classification'
    mean, std = mc_predict(
        model, test['x'],
        mc_passes=train_cfg.mc_passes, batch_size=train_cfg.batch_size,
        apply_sigmoid=is_classification,
    )
    print(f'Test prob: mean={mean.mean():.4f} max={mean.max():.4f} '
          f'std mean={std.mean():.4f}')

    val_mean, val_std = mc_predict(
        model, val['x'],
        mc_passes=train_cfg.mc_passes, batch_size=train_cfg.batch_size,
        apply_sigmoid=is_classification,
    )

    predictions = build_predictions_frame(
        test['timestamp'], test['ticker'], mean, std, cfg.data.horizons,
    )
    val_predictions = build_predictions_frame(
        val['timestamp'], val['ticker'], val_mean, val_std, cfg.data.horizons,
    )

    # 3) Bayes-калибровка (без Šidák).
    cost_per_trade = 2.0 * (cfg.trading.commission_rate + cfg.trading.slippage_rate)
    lr_targets = attach_lr_targets(prepared.full_frame, val, cfg.data.horizons)
    bayes_calib = calibrate_bayes_threshold(
        val_predictions, lr_targets, cost_per_trade=cost_per_trade,
    )
    trading_cfg_bayes = dataclasses.replace(
        cfg.trading,
        probability_threshold=bayes_calib.min_expected_return,
        selection_correction='none',
    )
    bayes_signals = SignalGenerator(trading_cfg_bayes, mode=cfg.data.mode).generate(predictions)
    bayes_bt = run_backtest(bayes_signals, test_prices, trading_cfg_bayes)
    bayes_m = compute_metrics(bayes_bt.equity, bayes_bt.trades, periods_per_year=bars_per_year)
    print(f'Bayes T={bayes_calib.min_expected_return:.4f}, '
          f'BUY={int((bayes_signals["action"]=="BUY").sum())}, '
          f'final={bayes_m["final_equity"]:,.0f}, sharpe={bayes_m["sharpe"]:.3f}')

    # 4) Conformal directional.
    val_targets_conf = attach_actual_targets(val, cfg.data.horizons)
    conf_gen = ConformalSignalGenerator(cfg.trading, alpha=0.1, directional=True)
    conf_calib = conf_gen.calibrate(val_predictions, val_targets_conf)
    conf_signals = conf_gen.generate(predictions)
    conf_bt = run_backtest(conf_signals, test_prices, cfg.trading)
    conf_m = compute_metrics(conf_bt.equity, conf_bt.trades, periods_per_year=bars_per_year)
    print(f'Conformal[dir] T={conf_calib.threshold:.4f}, '
          f'BUY={int((conf_signals["action"]=="BUY").sum())}, '
          f'final={conf_m["final_equity"]:,.0f}, sharpe={conf_m["sharpe"]:.3f}')

    # 5) Random monkeys (один раз для каждой архитектуры — разные сигналы дают
    # разный trade_probability, поэтому бенчмарк свой).
    n_buy = int((bayes_signals['action'] == 'BUY').sum())
    if n_buy > 0:
        avg_h = int(round(bayes_bt.trades['horizon'].mean())) if not bayes_bt.trades.empty else int(np.mean(cfg.data.horizons))
        n_bars = int(len(test_prices.index.unique()))
        trade_prob = max(min(n_buy / max(n_bars * cfg.trading.max_positions, 1), 1.0), 1e-4)
        report = run_random_portfolios(
            test_prices, cfg.trading,
            avg_horizon=avg_h, trade_probability=trade_prob,
            strategy_final=bayes_m['final_equity'],
            seed=cfg.training.seed,
        )
        z_bayes = report.strategy_z_score
        sig_bayes = report.is_significant
    else:
        z_bayes, sig_bayes = float('nan'), False

    return {
        'arch': arch,
        'n_params': n_params,
        'n_trainable': n_trainable,
        'best_epoch': best_ep,
        'best_val_loss': best_val,
        'prob_mean': float(mean.mean()),
        'prob_max': float(mean.max()),
        'prob_std_mean': float(std.mean()),
        'bayes_T': float(bayes_calib.min_expected_return),
        'bayes_n_buy': int((bayes_signals['action'] == 'BUY').sum()),
        'bayes_final_equity': float(bayes_m['final_equity']),
        'bayes_total_return': float(bayes_m['total_return']),
        'bayes_sharpe': float(bayes_m['sharpe']),
        'bayes_max_dd': float(bayes_m['max_drawdown']),
        'bayes_z_score': float(z_bayes),
        'bayes_significant': bool(sig_bayes),
        'conf_T': float(conf_calib.threshold),
        'conf_n_buy': int((conf_signals['action'] == 'BUY').sum()),
        'conf_final_equity': float(conf_m['final_equity']),
        'conf_total_return': float(conf_m['total_return']),
        'conf_sharpe': float(conf_m['sharpe']),
        'bayes_equity': bayes_bt.equity,
        'conf_equity': conf_bt.equity,
    }


## 4. Прогон трёх архитектур

DLinear и NLinear — лёгкие, обучаются за минуты. MOMENT тяжелее
(125M параметров, encoder заморожен), но fit быстрый — backprop
только через голову. **Если нет интернета или momentfm не установлен,
ячейку MOMENT можно пропустить.**

In [ ]:
results = {}

# --- VLinear (vanilla Linear) ---
results['vlinear'] = train_and_evaluate('vlinear')

# --- XLinear (arXiv:2601.09237) ---
results['xlinear'] = train_and_evaluate('xlinear')


In [ ]:
# Установка momentfm — отдельной ячейкой, чтобы её можно было
# перезапустить в случае проблем с зависимостями (типичный
# конфликт torch/transformers в свежем Colab). Если MOMENT-ячейка
# ниже падает с ImportError — запусти эту, дождись Successfully
# installed, потом снова MOMENT-ячейку.
!pip install -q momentfm transformers huggingface_hub


In [ ]:
# --- MOMENT (требует momentfm + интернет на первый прогон) ---
try:
    results['moment'] = train_and_evaluate(
        'moment',
        # Уменьшаем batch для MOMENT: encoder тяжёлый, активации больше.
        train_overrides={'batch_size': 256, 'mc_passes': 30, 'epochs': 20},
    )
except Exception as exc:
    print(f'MOMENT не запустился: {exc!r}')
    print('Если это про momentfm — выполни `pip install momentfm` и перезапусти ячейку.')


## 5. Сравнительная таблица

In [ ]:
def _summary_row(r: dict) -> dict:
    return {
        'arch': r['arch'],
        'params': r['n_params'],
        'trainable': r['n_trainable'],
        'best_epoch': r['best_epoch'],
        'best_val': r['best_val_loss'],
        'prob_max': r['prob_max'],
        'prob_std_mean': r['prob_std_mean'],
        'bayes_T': r['bayes_T'],
        'bayes_buy': r['bayes_n_buy'],
        'bayes_ret%': r['bayes_total_return'] * 100,
        'bayes_sharpe': r['bayes_sharpe'],
        'bayes_z': r['bayes_z_score'],
        'bayes_sig': r['bayes_significant'],
        'conf_T': r['conf_T'],
        'conf_buy': r['conf_n_buy'],
        'conf_ret%': r['conf_total_return'] * 100,
        'conf_sharpe': r['conf_sharpe'],
    }


summary = pd.DataFrame([_summary_row(r) for r in results.values()])
print('=== СРАВНЕНИЕ АРХИТЕКТУР ===')
print(summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


In [ ]:
# --- Equity-кривые сравнения ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
colors = {'vlinear': '#1f77b4', 'xlinear': '#2ca02c', 'moment': '#d62728'}

for arch, r in results.items():
    eq = r['bayes_equity']
    if not eq.empty:
        ax1.plot(eq.index, eq.values, label=arch, color=colors.get(arch, 'gray'), linewidth=1.2)
    eq2 = r['conf_equity']
    if not eq2.empty:
        ax2.plot(eq2.index, eq2.values, label=arch, color=colors.get(arch, 'gray'), linewidth=1.2)

for ax, title in [(ax1, 'Bayes filter'), (ax2, 'Conformal directional')]:
    ax.axhline(cfg.trading.initial_capital, color='gray', linestyle=':', alpha=0.6)
    ax.set_title(title); ax.set_xlabel('дата'); ax.set_ylabel('капитал, RUB')
    ax.legend(loc='best', fontsize=9)

fig.autofmt_xdate(); fig.tight_layout(); plt.show()


## Что дальше

- Лучшая архитектура по `bayes_sharpe` (или `bayes_total_return`) — берётся
  как baseline для дальнейшей валидации.
- Если **MOMENT** обыграл TimeXer/XLinear/VLinear — это сильный аргумент в
  ВКР: foundation-features времени уже содержат полезную репрезентацию,
  и наша задача сводится к небольшому fine-tune головы. Это спасает
  ситуацию маленьких датасетов (108k сэмплов).
- Если **XLinear** или **VLinear** оказались сильнее трансформеров —
  тоже defendable: значит, на наших фичах нелинейные модели
  переобучаются, и для production стоит закрепить лёгкий MLP.
- Threshold sweep (как в `training_pipeline.ipynb` §11) можно прогнать
  на лучшей модели — копировать ячейку оттуда.
